In [174]:
from abc import ABC, abstractmethod
from collections import namedtuple

Customer = namedtuple('Customer', 'name fidelity')

In [175]:
park = Customer('Park', 100)
print(park)

Customer(name='Park', fidelity=100)


In [176]:
class LineItem:
    def __init__(self, product, quantity, price):
        self.product = product
        self.quantity = quantity
        self.price = price

    def total(self):
        return self.price * self.quantity

In [177]:
class Order:
    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion

    def total(self):
        if not hasattr(self, '_total'):
            self._total = sum(item.total() for item in self.cart)
        return self._total

    def due(self):
        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion.discount(self)
        return self.total() - discount

    def __repr__(self):
        fmt = '<Order total: {:.2f} due: {:.2f}>'
        return fmt.format(self.total(), self.due())

In [178]:
class Promotion(ABC):

    @abstractmethod
    def discount(self, order):
        pass

In [179]:
class FidelityPromo(Promotion):
    def discount(self, order):
        return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0

class BulkItemPromo(Promotion):
    def discount(self, order):
        discount = 0
        for item in order.cart:
            if item.quantity >= 20:
                discount += item.total() * 0.1
        return discount

class LargeOrderPromo(Promotion):
    def discount(self, order):
        distinct_items = {item.product for item in order.cart}
        if len(distinct_items) >= 10:
            return order.total() * 0.07
        return 0

In [180]:
joe = Customer('John Doe', 0)
ann = Customer('Ann Smith', 1100)

cart = [LineItem('banana', 4, .5),
        LineItem('apple', 10, 1.5),
        LineItem('watermellon', 5, 5.0)]

In [181]:
Order(joe, cart, FidelityPromo())

<Order total: 42.00 due: 42.00>

In [182]:
Order(ann, cart, FidelityPromo())

<Order total: 42.00 due: 39.90>

In [183]:
banana_cart = [LineItem('banana', 30, .5),
               LineItem('apple', 10, 1.5)]

In [184]:
Order(joe, banana_cart, BulkItemPromo())

<Order total: 30.00 due: 28.50>

In [185]:
long_order = [LineItem(str(item_code), 1, 1.0) for item_code in range(10)]

In [186]:
Order(joe, long_order, LargeOrderPromo())

<Order total: 10.00 due: 9.30>

In [187]:
Order(joe, cart, LargeOrderPromo())

<Order total: 42.00 due: 42.00>

In [188]:
class Order:
    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion

    def total(self):
        if not hasattr(self, '_total'):
            self._total = sum(item.total() for item in self.cart)
        return self._total

    def due(self):
        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion(self)
        return self.total() - discount

    def __repr__(self):
        fmt = '<Order total: {:.2f} due: {:.2f}>'
        return fmt.format(self.total(), self.due())

In [189]:
def fidelity_promo(order):
    return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0

def bulk_item_promo(order):
    discount = 0
    for item in order.cart:
        if item.quantity >= 20:
            discount += item.total() * 0.1
    return discount

def large_order_promo(order):
    distinct_items = {item.product for item in order.cart}
    if len(distinct_items) >= 10:
        return order.total() * 0.07
    return 0

In [190]:
Order(joe, cart, fidelity_promo)

<Order total: 42.00 due: 42.00>

In [191]:
Order(ann, cart, fidelity_promo)

<Order total: 42.00 due: 39.90>

In [192]:
Order(joe, banana_cart, bulk_item_promo)

<Order total: 30.00 due: 28.50>

In [193]:
Order(ann, long_order, large_order_promo)

<Order total: 10.00 due: 9.30>

In [194]:
promos = [fidelity_promo, bulk_item_promo, large_order_promo]

def best_promo(order):
    return max(promo(order) for promo in promos)

In [195]:
Order(joe, long_order, best_promo)

<Order total: 10.00 due: 9.30>

In [196]:
Order(joe, banana_cart, best_promo)

<Order total: 30.00 due: 28.50>

In [197]:
Order(ann, cart, best_promo)

<Order total: 42.00 due: 39.90>

In [198]:
globals().keys()

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'ABC', 'abstractmethod', 'namedtuple', 'Customer', '_i2', 'park', '_i3', '_i4', '_i5', 'LineItem', '_i6', 'Order', '_i7', 'Promotion', '_i8', 'FidelityPromo', 'BulkItemPromo', 'LargeOrderPromo', '_i9', 'joe', 'ann', 'cart', '_i10', '_i11', '_i12', 'banana_cart', '_i13', '_i14', 'long_order', '_i15', '_i16', '_i17', '_i18', 'fidelity_promo', 'bulk_item_promo', 'large_order_promo', '_i19', '_i20', '_i21', '_i22', '_i23', 'promos', 'best_promo', '_i24', '_i25', '_i26', '_i27', '_i28', '_i29', '_i30', '_i31', '_i32', '_i33', '_i34', '_i35', '_i36', '_i37', '_i38', '_i39', '_i40', '_i41', '_i42', '_i43', '_i44', '_i45', '_i46', '_i47', '_i48', '_i49', '_i50', '_i51', '_i52', '_i53', '_i54', '_i55', '_i56', '_i57', '_i58', '_i59', '_i60', '_60', '_i61', '_i62', '_i63', '_i64', '

In [199]:
globals()['best_promo']

<function __main__.best_promo(order)>

In [200]:
promos = [globals()[name] for name in globals()
            if name.endswith('_promo')
            and name != 'best_promo']

In [201]:
promos

[<function __main__.fidelity_promo(order)>,
 <function __main__.bulk_item_promo(order)>,
 <function __main__.large_order_promo(order)>]